In [ ]:
from IPython.display import clear_output

In [ ]:
# %pip install nltk tqdm

clear_output()

# Content:

In this demo, we will build an N-gram probabilistic based Language model

We will use NLTK library to download the dataset and handle our text.

In [ ]:
import random

import nltk
from nltk.util import ngrams as build_ngrams
from nltk.tokenize import word_tokenize
from collections import defaultdict
from tqdm import tqdm

nltk.download('punkt')

## Downloading the dataset

In [ ]:
# Download the IMDB dataset
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews

In [ ]:
tokenized_data = movie_reviews.sents()  # sents is sentences (not full reviews as can be very long). They are alread tokenized.

## Building the model

In [ ]:
sos_token = '<SOS>'  # start of sentence token. Appending this at the start will make the selection of first token also probabilistic based according to corpus
eos_token = '<EOS>'  # to indicate a sentence has ended and we should stop generating

In [ ]:
class NGramLanguageModel():

    def __init__(self, n):

        self.n = n
        self.word_freqs = defaultdict(dict) # 1 to n-1 grams: {dict of possible words: frequency}

    def train(self, sentences):

        for sentence in tqdm(sentences, desc='Processing Sentences'):

            for gram_size in range(2, self.n+1):  # 2 because we need to make key(gram[:-1]) val(gram[-1]) pairs and need atleast 2.

                ngrams = build_ngrams([sos_token]+sentence+[eos_token], gram_size)  # need to manually append eos_token at the end of sentences

                for ngram in ngrams:

                    key = ngram[:-1]
                    value = ngram[-1]

                    self.word_freqs[key][value] = self.word_freqs[key].get(value, 0)+1  # if key doesn't exist already then freq is 0. Whatever the frequency is, add 1 to it.

    def generate_sentence(self, starting_state=None, max_length=50):

        generated_sentence = []

        if starting_state is None:
            generated_sentence = [sos_token]
        elif isinstance(starting_state, str):
            generated_sentence = starting_state.split()

        if generated_sentence[0] != sos_token:
            generated_sentence = [sos_token]+generated_sentence

        max_key_len = self.n-1

        if tuple(generated_sentence[-1:]) not in self.word_freqs:  # python automatically takes care of the case if the max_key_len is bigger than total list size
            raise ValueError('Invalid starting state')

        while len(generated_sentence) <= max_length:

            for key_len in range(max_key_len, 0, -1):  # for loop for the condition: if we can't find a combination of lets say the latest 5 gram in corpus, we go for 4 then 3 and so on
                last_tokens = generated_sentence[-key_len:]
                next_word_freqs = self.word_freqs[tuple(last_tokens)]
                if len(next_word_freqs) > 0:
                    break

            words, freqs = list(zip(*next_word_freqs.items()))  # [words...], [freqs..]

            next_word = random.choices(words, weights=freqs, k=1)[0]  # no need to divide and convert to probability first. choices() can take weights as it is.
            generated_sentence.append(next_word)

            if next_word == eos_token:
                break

        return generated_sentence


In [ ]:
model = NGramLanguageModel(n=5)  # The bigger the value of n, the better but bigger the model.

In [ ]:
model.train(tokenized_data)

## Let's see the results

In [ ]:
# 5 completely random sentences
for _ in range(5):
    sentence_tokens = model.generate_sentence()
    print(' '.join(sentence_tokens))  # skip SOS and EOS tokens
    print('-'*20)

In [ ]:
model.generate_sentence(starting_state='I felt like')